In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.home() / "aspect_sentiment"

# 모델 불러오기
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_PATH = PROJECT_ROOT / "models/sentiment"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model = model.to(device)
model.eval()

print("모델 불러오기 완료")
print("사용 장치:", device)
print("라벨:", model.config.id2label)

In [ ]:
# 문장 넣어 예측 하는 함수
import torch
import torch.nn.functional as F

def predict_sentiment(aspect, text):
    model_input = f"[속성] {aspect} [문장] {text}"

    inputs = tokenizer(
        model_input,
        return_tensors="pt",
        truncation=True,
        max_length=64,
        padding=True
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = F.softmax(outputs.logits, dim=-1)
        predicted_id = torch.argmax(probabilities, dim=-1).item()

    return {
        "aspect": aspect,
        "text": text,
        "label_id": predicted_id,
        "label": model.config.id2label[predicted_id],
        "confidence": probabilities[0][predicted_id].item(),
        "probabilities": {
            model.config.id2label[i]: probabilities[0][i].item()
            for i in range(model.config.num_labels)
        }
    }

In [ ]:
result = predict_sentiment(
    aspect="발림성",
    text="피부에 부드럽게 잘 발려요"
)

result

In [ ]:
test_cases = [
    ("발림성", "피부에 부드럽게 잘 발려요"),
    ("발림성", "너무 뻑뻑해서 잘 발리지 않아요"),
    ("기능/효과", "아직 효과는 잘 모르겠어요"),
    ("보습력/수분감", "엄청 촉촉하지는 않지만 건조하지도 않아요")
]

for aspect, text in test_cases:
    result = predict_sentiment(aspect, text)
    print(f"[{aspect}] {text}")
    print(
        f"예측: {result['label']} "
        f"({result['confidence']:.4f})"
    )
    print(result["probabilities"])
    print()

In [ ]:
# 검증 데이터 불러오기
import pandas as pd

validation_df = pd.read_csv("./data/sentiment/validation.csv")

print("Validation 데이터 수:", len(validation_df))
display(validation_df.head())

In [ ]:
# 검증 예측
import numpy as np
from torch.utils.data import DataLoader, TensorDataset

texts = (
    "[속성] " + validation_df["aspect"].astype(str)
    + " [문장] " + validation_df["sentiment_text"].astype(str)
).tolist()

encoded = tokenizer(
    texts,
    padding=True,
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

dataset = TensorDataset(encoded["input_ids"], encoded["attention_mask"])
loader = DataLoader(dataset, batch_size=64)

predictions = []
confidences = []

model.eval()

with torch.no_grad():
    for input_ids, attention_mask in loader:
        outputs = model(
            input_ids=input_ids.to(device),
            attention_mask=attention_mask.to(device)
        )
        probabilities = torch.softmax(outputs.logits, dim=-1)
        batch_confidence, batch_predictions = probabilities.max(dim=-1)

        predictions.extend(batch_predictions.cpu().tolist())
        confidences.extend(batch_confidence.cpu().tolist())

validation_df["predicted_label"] = predictions
validation_df["confidence"] = confidences
validation_df["predicted_sentiment"] = validation_df["predicted_label"].map(
    {0: "긍정", 1: "부정", 2: "중립"}
)

print("Validation 전체 예측 완료")

In [ ]:
display(validation_df[
    ["aspect", "sentiment_text", "label", "predicted_sentiment", "confidence"]
].head(30))

In [ ]:
label_names = {0: "긍정", 1: "부정", 2: "중립"}
validation_df["true_sentiment"] = validation_df["label"].map(label_names)

In [ ]:
wrong_df = validation_df[
    validation_df["label"] != validation_df["predicted_label"]
].copy()

display(wrong_df[
    [
        "aspect",
        "sentiment_text",
        "true_sentiment",
        "predicted_sentiment",
        "confidence"
    ]
].head(30))

In [ ]:
# 오분류 결과 저장
wrong_df.to_csv(
    "./data/reports/sentiment_wrong_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

print("오분류 결과 저장 완료:", len(wrong_df))

In [ ]:
display(wrong_df[
    ["aspect", "sentiment_text", "raw_text", "true_sentiment",
     "predicted_sentiment", "confidence"]
].head(30))

In [ ]:
high_confidence_wrong = wrong_df[wrong_df["confidence"] >= 0.95].copy()

print("고확신 오분류 수:", len(high_confidence_wrong))

display(high_confidence_wrong[
    ["aspect", "sentiment_text", "raw_text", "true_sentiment",
     "predicted_sentiment", "confidence"]
].head(50))

In [ ]:
# 어떤 답을 어떻게 잘못 예측 했는지 확인
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

label_names = ["긍정", "부정", "중립"]

cm = confusion_matrix(
    validation_df["label"],
    validation_df["predicted_label"],
    labels=[0, 1, 2]
)

cm_df = pd.DataFrame(
    cm,
    index=[f"실제_{name}" for name in label_names],
    columns=[f"예측_{name}" for name in label_names]
)

display(cm_df)

In [ ]:
cm_ratio = cm / cm.sum(axis=1, keepdims=True)

cm_ratio_df = pd.DataFrame(
    cm_ratio,
    index=[f"실제_{name}" for name in label_names],
    columns=[f"예측_{name}" for name in label_names]
).round(4)

display(cm_ratio_df)

In [ ]:
# 속성별 성능 확인
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd

aspect_results = []

for aspect, group in validation_df.groupby("aspect"):
    if len(group) < 5:
        continue

    aspect_results.append({
        "aspect": aspect,
        "count": len(group),
        "accuracy": accuracy_score(group["label"], group["predicted_label"]),
        "macro_f1": f1_score(
            group["label"],
            group["predicted_label"],
            labels=[0, 1, 2],
            average="macro",
            zero_division=0
        ),
        "wrong_count": (group["label"] != group["predicted_label"]).sum()
    })

aspect_result_df = pd.DataFrame(aspect_results).sort_values(
    ["macro_f1", "count"],
    ascending=[True, False]
)

display(aspect_result_df.head(30))

In [ ]:
aspect_result_df["wrong_ratio"] = (
    aspect_result_df["wrong_count"] / aspect_result_df["count"]
)

display(
    aspect_result_df[
        ["aspect", "count", "accuracy", "macro_f1", "wrong_count", "wrong_ratio"]
    ].head(30)
)